In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial.distance import cdist

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = "dummy_data_50"
RNA_DIR = os.path.join(OUTPUT_DIR, "rna")
MSI_DIR = os.path.join(OUTPUT_DIR, "msi")

GROUPS = ["YC", "YAD", "AC", "AAD"]
SAMPLES_PER_GROUP = 4

# Visium-like parameters: 55um spot diameter, 100um center-to-center distance
RNA_SPOT_DIAMETER = 55
RNA_SPOT_SPACING = 100  # Center-to-center distance

MSI_PIXEL_SIZE = 60  # Cartesian grid

# 50 Ground Truth Patterns - organized by category
GT_PATTERNS = [
    # === GRADIENTS (6 patterns) ===
    "Gradient_X",              # Left to right
    "Gradient_Y",              # Bottom to top
    "Gradient_Diagonal_NE",    # Southwest to Northeast
    "Gradient_Diagonal_NW",    # Southeast to Northwest
    "Gradient_Radial_In",      # Center bright, edges dark
    "Gradient_Radial_Out",     # Edges bright, center dark
    
    # === WAVES & STRIPES (8 patterns) ===
    "Stripes_Vertical",        # Vertical sinusoidal
    "Stripes_Horizontal",      # Horizontal sinusoidal
    "Stripes_Diagonal_45",     # 45-degree stripes
    "Stripes_Diagonal_135",    # 135-degree stripes
    "Waves_Concentric",        # Concentric circles (bullseye)
    "Waves_Spiral",            # Spiral pattern
    "Waves_Interference",      # Two-source interference
    "Waves_Ripple",            # Decaying ripples from center
    
    # === BLOBS & SPOTS (10 patterns) ===
    "Blob_Center",             # Gaussian blob center
    "Blob_TopRight",           # Gaussian blob top-right
    "Blob_TopLeft",            # Gaussian blob top-left
    "Blob_BottomRight",        # Gaussian blob bottom-right
    "Blob_BottomLeft",         # Gaussian blob bottom-left
    "Spots_Grid_Dense",        # Dense polka dots
    "Spots_Grid_Sparse",       # Sparse polka dots
    "Spots_Random_Large",      # Large random spots
    "Spots_Triangular",        # Triangular arrangement
    "Spots_Hexagonal",         # Hexagonal arrangement
    
    # === RINGS & DONUTS (6 patterns) ===
    "Ring_Inner",              # Inner ring
    "Ring_Outer",              # Outer ring
    "Ring_Double",             # Double concentric rings
    "Ring_Eccentric",          # Off-center ring
    "Ring_Elliptical",         # Elliptical ring
    "Ring_Partial",            # Partial arc/ring
    
    # === GEOMETRIC PATTERNS (8 patterns) ===
    "Checkerboard_Fine",       # Fine checkerboard
    "Checkerboard_Coarse",     # Coarse checkerboard
    "Quadrant_Alternating",    # Alternating quadrants
    "Sectors_4",               # 4 pie sectors
    "Sectors_8",               # 8 pie sectors
    "Triangle_Pattern",        # Triangular tessellation
    "Diamond_Pattern",         # Diamond/rhombus pattern
    "Honeycomb",               # Honeycomb hexagonal
    
    # === COMPLEX BIOLOGICAL-LIKE (12 patterns) ===
    "Cortical_Layers",         # Layered like cortex
    "Hotspot_Cluster",         # Multiple clustered hotspots
    "Edge_Enhancement",        # Enhanced at tissue edges
    "Core_Shell",              # Core-shell structure
    "Branching",               # Branching/dendritic pattern
    "Laminar_Curved",          # Curved laminar structure
    "Mosaic_Irregular",        # Irregular mosaic
    "Gradient_Sigmoid",        # Sigmoid transition
    "Bimodal_Distribution",    # Two distinct regions
    "Punctate_Dense",          # Dense punctate pattern
    "Periventricular",         # Around a central void
    "Asymmetric_Lobe",         # Asymmetric lobed structure
]

NOISE_FEATURES = 500  # Reduced but more realistic noise features

os.makedirs(RNA_DIR, exist_ok=True)
os.makedirs(MSI_DIR, exist_ok=True)
np.random.seed(42)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def generate_halfbrain_shape(sample_seed, width=3000, height=3000):
    """
    Generates a half-brain shaped tissue mask resembling mouse coronal sections
    at approximately Bregma -2.24mm, based on actual H&E histology images.
    
    Realistic dimensions:
    - Max height: 4.9-5.4 mm (4900-5400 µm)
    - Max width: 4.4-5.0 mm (4400-5000 µm)  
    - Height/Width ratio: 1.0 to 1.2
    - Max width location: between 1/2 and 9/10 of the height from bottom
    - Bottom width: 70% of max width
    - Left side: straight vertical line (midline cut)
    - Bottom: straight horizontal line
    - Right and top: smooth D-curve
    
    Orientation:
    - Medial (midline) on LEFT - straight vertical cut
    - Lateral on RIGHT - curved
    - Dorsal at TOP - curved
    - Ventral at BOTTOM - straight, narrower
    """
    np.random.seed(sample_seed)
    
    # Random variation for realistic dimensions
    # Max height: 4900-5400 µm
    brain_height = np.random.uniform(4900, 5400)
    
    # Height/Width ratio: 1.0-1.2
    hw_ratio = np.random.uniform(1.0, 1.2)
    brain_max_width = brain_height / hw_ratio
    
    # Clamp max width to 4400-5000 µm range
    brain_max_width = np.clip(brain_max_width, 4400, 5000)
    
    # Max width location: between 0.5 and 0.9 of height from bottom
    max_width_fraction = np.random.uniform(0.5, 0.9)
    
    # Small random variations
    rotation = np.random.uniform(-0.05, 0.05)  # up to ~3 degrees
    offset_x = np.random.uniform(-50, 50)
    offset_y = np.random.uniform(-50, 50)
    
    # Position brain - center it in the field with some margin
    # Left edge (medial) positioned to center the brain horizontally
    left_edge = (width - brain_max_width) / 2 + offset_x
    # Bottom edge positioned to center vertically  
    bottom_edge = (height - brain_height) / 2 + offset_y
    
    # Key coordinates
    bottom_width = brain_max_width * 0.70  # Bottom is 70% of max width
    max_width_y = bottom_edge + brain_height * max_width_fraction
    top_edge = bottom_edge + brain_height
    
    def is_in_halfbrain(x, y):
        # Apply rotation around the center of the brain
        cx = left_edge + brain_max_width * 0.5
        cy = bottom_edge + brain_height * 0.5
        
        dx = x - cx
        dy = y - cy
        x_rot = dx * np.cos(rotation) - dy * np.sin(rotation) + cx
        y_rot = dx * np.sin(rotation) + dy * np.cos(rotation) + cy
        
        # Check basic bounds first
        # Left boundary: straight vertical line
        if x_rot < left_edge:
            return False
        
        # Bottom boundary: straight horizontal line
        if y_rot < bottom_edge:
            return False
        
        # Top boundary
        if y_rot > top_edge:
            return False
        
        # Normalize y position within the brain (0 = bottom, 1 = top)
        y_norm = (y_rot - bottom_edge) / brain_height
        y_norm = np.clip(y_norm, 0, 1)
        
        # Calculate the right boundary as a function of height
        # - At y_norm=0 (bottom): width = bottom_width (70% of max)
        # - At y_norm=max_width_fraction: width = brain_max_width (100%)
        # - At y_norm=1 (top): curves back to 0
        
        if y_norm <= max_width_fraction:
            # From bottom to max width: smooth curve outward
            t = y_norm / max_width_fraction  # 0 to 1 over this range
            # Smooth interpolation using sine
            right_width = bottom_width + (brain_max_width - bottom_width) * np.sin(t * np.pi / 2)
        else:
            # From max width to top: curve inward (the dome)
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)  # 0 to 1
            # Quarter ellipse curve back to 0
            right_width = brain_max_width * np.sqrt(max(0, 1 - t**2))
        
        right_edge = left_edge + right_width
        
        if x_rot > right_edge:
            return False
        
        # Handle the dome region (top portion)
        if y_norm > max_width_fraction:
            # x position relative to the brain width at this height
            x_rel = (x_rot - left_edge) / brain_max_width
            x_rel = np.clip(x_rel, 0, 1)
            
            # The dome follows approximately a quarter ellipse
            t = (y_norm - max_width_fraction) / (1 - max_width_fraction)
            
            # At the medial edge (x_rel near 0), allow more height
            # At lateral edge, height is constrained by the dome curve
            if x_rel < 0.05:
                max_y_norm = 1.0
            else:
                # Elliptical constraint
                max_y_norm = max_width_fraction + (1 - max_width_fraction) * np.sqrt(max(0, 1 - x_rel**2))
            
            if y_norm > max_y_norm:
                return False
        
        return True
    
    return is_in_halfbrain


def generate_pattern_values(coords, pattern_type, width=3000, height=3000, add_noise=False, noise_seed=None):
    """
    Generates intensity values (0-1) based on spatial coordinates.
    
    Parameters:
    -----------
    coords : numpy array
        Nx2 array of (x, y) coordinates
    pattern_type : str
        Name of the pattern to generate
    width, height : int
        Field dimensions
    add_noise : bool
        Whether to add small random noise (default False for exact matching)
    noise_seed : int or None
        Seed for noise generation (if add_noise=True)
    """
    x, y = coords[:, 0], coords[:, 1]
    n = len(x)
    val = np.zeros(n)
    
    # Normalized coordinates (0-1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-8)
    cx, cy = x.mean(), y.mean()
    
    # Distance from center
    dist_center = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_dist = dist_center.max() + 1e-8
    dist_norm = dist_center / max_dist
    
    # Angle from center
    angle = np.arctan2(y - cy, x - cx)
    
    # Use a deterministic seed for patterns that need randomness
    # This ensures the same pattern produces identical results regardless of when it's called
    pattern_seed = hash(pattern_type) % 2**31
    
    # === GRADIENTS ===
    if pattern_type == "Gradient_X":
        val = x_norm
        
    elif pattern_type == "Gradient_Y":
        val = y_norm
        
    elif pattern_type == "Gradient_Diagonal_NE":
        val = (x_norm + y_norm) / 2
        
    elif pattern_type == "Gradient_Diagonal_NW":
        val = (1 - x_norm + y_norm) / 2
        
    elif pattern_type == "Gradient_Radial_In":
        val = 1 - dist_norm
        
    elif pattern_type == "Gradient_Radial_Out":
        val = dist_norm
        
    # === WAVES & STRIPES ===
    elif pattern_type == "Stripes_Vertical":
        val = (np.sin(x / 80) + 1) / 2
        
    elif pattern_type == "Stripes_Horizontal":
        val = (np.sin(y / 80) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_45":
        val = (np.sin((x + y) / 100) + 1) / 2
        
    elif pattern_type == "Stripes_Diagonal_135":
        val = (np.sin((x - y) / 100) + 1) / 2
        
    elif pattern_type == "Waves_Concentric":
        val = (np.sin(dist_center / 60) + 1) / 2
        
    elif pattern_type == "Waves_Spiral":
        spiral = angle + dist_center / 150
        val = (np.sin(spiral * 3) + 1) / 2
        
    elif pattern_type == "Waves_Interference":
        # Two point sources
        d1 = np.sqrt((x - width*0.3)**2 + (y - height*0.5)**2)
        d2 = np.sqrt((x - width*0.7)**2 + (y - height*0.5)**2)
        val = (np.sin(d1/50) + np.sin(d2/50) + 2) / 4
        
    elif pattern_type == "Waves_Ripple":
        val = np.exp(-dist_norm * 2) * (np.sin(dist_center / 40) + 1) / 2
        
    # === BLOBS & SPOTS ===
    elif pattern_type == "Blob_Center":
        val = np.exp(-dist_center**2 / (2 * (width*0.2)**2))
        
    elif pattern_type == "Blob_TopRight":
        bx, by = x.max() * 0.75, y.max() * 0.75
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_TopLeft":
        bx, by = x.min() + (x.max()-x.min()) * 0.25, y.max() * 0.75
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomRight":
        bx, by = x.max() * 0.75, y.min() + (y.max()-y.min()) * 0.25
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Blob_BottomLeft":
        bx, by = x.min() + (x.max()-x.min()) * 0.25, y.min() + (y.max()-y.min()) * 0.25
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * (width*0.15)**2))
        
    elif pattern_type == "Spots_Grid_Dense":
        sx = np.sin(x / 100)
        sy = np.sin(y / 100)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Grid_Sparse":
        sx = np.sin(x / 200)
        sy = np.sin(y / 200)
        val = np.clip(sx * sy, 0, 1)
        
    elif pattern_type == "Spots_Random_Large":
        # Multiple random Gaussian blobs - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(5):
            bx = rng.uniform(x.min(), x.max())
            by = rng.uniform(y.min(), y.max())
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.1)**2))
        val = val / val.max()
        
    elif pattern_type == "Spots_Triangular":
        # Triangular grid of spots
        spacing = 400
        val = np.zeros(n)
        for i in range(10):
            for j in range(10):
                bx = i * spacing + (j % 2) * (spacing / 2)
                by = j * spacing * 0.866
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 80**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Spots_Hexagonal":
        spacing = 350
        val = np.zeros(n)
        for i in range(12):
            for j in range(12):
                bx = i * spacing + (j % 2) * (spacing / 2)
                by = j * spacing * 0.866
                dist = np.sqrt((x - bx)**2 + (y - by)**2)
                val += np.exp(-dist**2 / (2 * 60**2))
        val = np.clip(val, 0, 1)
        
    # === RINGS & DONUTS ===
    elif pattern_type == "Ring_Inner":
        target_r = max_dist * 0.2
        val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.04)**2))
        
    elif pattern_type == "Ring_Outer":
        target_r = max_dist * 0.7
        val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Double":
        r1 = max_dist * 0.25
        r2 = max_dist * 0.55
        val = np.exp(-((dist_center - r1)**2) / (2 * (width*0.03)**2)) + \
              np.exp(-((dist_center - r2)**2) / (2 * (width*0.03)**2))
        val = val / val.max()
        
    elif pattern_type == "Ring_Eccentric":
        off_cx = cx + width * 0.15
        off_cy = cy + height * 0.1
        dist_off = np.sqrt((x - off_cx)**2 + (y - off_cy)**2)
        target_r = max_dist * 0.35
        val = np.exp(-((dist_off - target_r)**2) / (2 * (width*0.05)**2))
        
    elif pattern_type == "Ring_Elliptical":
        a, b = max_dist * 0.5, max_dist * 0.3
        ellipse_dist = np.sqrt(((x - cx)/a)**2 + ((y - cy)/b)**2)
        val = np.exp(-((ellipse_dist - 1)**2) / (2 * 0.1**2))
        
    elif pattern_type == "Ring_Partial":
        target_r = max_dist * 0.4
        ring_val = np.exp(-((dist_center - target_r)**2) / (2 * (width*0.05)**2))
        # Only show part of the ring (angular mask)
        angle_mask = (angle > -np.pi/2) & (angle < np.pi/2)
        val = ring_val * angle_mask
        
    # === GEOMETRIC PATTERNS ===
    elif pattern_type == "Checkerboard_Fine":
        square_size = 200
        x_idx = (x // square_size).astype(int)
        y_idx = (y // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Checkerboard_Coarse":
        square_size = 500
        x_idx = (x // square_size).astype(int)
        y_idx = (y // square_size).astype(int)
        val = ((x_idx + y_idx) % 2).astype(float)
        
    elif pattern_type == "Quadrant_Alternating":
        q1 = (x > cx) & (y > cy)
        q3 = (x < cx) & (y < cy)
        val = (q1 | q3).astype(float)
        
    elif pattern_type == "Sectors_4":
        sector = ((angle + np.pi) / (np.pi/2)).astype(int) % 4
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Sectors_8":
        sector = ((angle + np.pi) / (np.pi/4)).astype(int) % 8
        val = (sector % 2).astype(float)
        
    elif pattern_type == "Triangle_Pattern":
        # Triangular wave pattern
        period = 300
        tri_x = np.abs((x % period) - period/2) / (period/2)
        tri_y = np.abs((y % period) - period/2) / (period/2)
        val = (tri_x + tri_y) / 2
        
    elif pattern_type == "Diamond_Pattern":
        period = 400
        val = (np.abs((x % period) - period/2) + np.abs((y % period) - period/2)) / period
        
    elif pattern_type == "Honeycomb":
        # Hexagonal pattern using distance to nearest hex center
        spacing = 200
        val = np.zeros(n)
        for i in range(-2, 20):
            for j in range(-2, 20):
                hx = i * spacing + (j % 2) * (spacing / 2)
                hy = j * spacing * 0.866
                dist = np.sqrt((x - hx)**2 + (y - hy)**2)
                # Create honeycomb walls (inverse of spots)
                val = np.maximum(val, 1 - np.clip(dist / (spacing * 0.4), 0, 1))
        val = 1 - val  # Invert to get honeycomb walls
        
    # === COMPLEX BIOLOGICAL-LIKE ===
    elif pattern_type == "Cortical_Layers":
        # Multiple concentric bands with varying intensities
        n_layers = 5
        layer_vals = [0.9, 0.3, 0.7, 0.2, 0.8]
        layer_idx = np.clip((dist_norm * n_layers).astype(int), 0, n_layers-1)
        val = np.array([layer_vals[i] for i in layer_idx])
        
    elif pattern_type == "Hotspot_Cluster":
        # Cluster of multiple hotspots - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        cluster_center_x = cx + rng.uniform(-width*0.1, width*0.1)
        cluster_center_y = cy + rng.uniform(-height*0.1, height*0.1)
        for _ in range(8):
            bx = cluster_center_x + rng.normal(0, width*0.08)
            by = cluster_center_y + rng.normal(0, height*0.08)
            dist = np.sqrt((x - bx)**2 + (y - by)**2)
            val += np.exp(-dist**2 / (2 * (width*0.05)**2))
        val = val / val.max()
        
    elif pattern_type == "Edge_Enhancement":
        # Higher at tissue edges (using distance from center as proxy)
        val = 1 - np.exp(-((dist_norm - 0.8)**2) / (2 * 0.15**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Core_Shell":
        # Core region + shell region
        core = np.exp(-dist_center**2 / (2 * (width*0.1)**2))
        shell = np.exp(-((dist_center - max_dist*0.5)**2) / (2 * (width*0.08)**2))
        val = 0.7 * core + 0.5 * shell
        val = val / val.max()
        
    elif pattern_type == "Branching":
        # Branching/dendritic pattern using polar coordinates
        n_branches = 5
        branch_angle = 2 * np.pi / n_branches
        branch_width = 0.3
        min_dist_to_branch = np.inf * np.ones(n)
        for i in range(n_branches):
            target_angle = i * branch_angle
            angle_diff = np.abs(np.mod(angle - target_angle + np.pi, 2*np.pi) - np.pi)
            min_dist_to_branch = np.minimum(min_dist_to_branch, angle_diff)
        val = np.exp(-min_dist_to_branch**2 / (2 * branch_width**2)) * (1 - np.exp(-dist_norm * 3))
        
    elif pattern_type == "Laminar_Curved":
        # Curved layers following a sinusoidal boundary
        curved_y = y - 200 * np.sin(x / 400)
        layer_idx = (curved_y // 300).astype(int) % 2
        val = layer_idx.astype(float)
        
    elif pattern_type == "Mosaic_Irregular":
        # Irregular mosaic using Voronoi-like pattern - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        n_seeds = 20
        seed_x = rng.uniform(x.min(), x.max(), n_seeds)
        seed_y = rng.uniform(y.min(), y.max(), n_seeds)
        seed_val = rng.rand(n_seeds)
        # Find nearest seed for each point
        nearest_idx = np.zeros(n, dtype=int)
        min_dist = np.inf * np.ones(n)
        for i in range(n_seeds):
            dist = np.sqrt((x - seed_x[i])**2 + (y - seed_y[i])**2)
            closer = dist < min_dist
            nearest_idx[closer] = i
            min_dist[closer] = dist[closer]
        val = seed_val[nearest_idx]
        
    elif pattern_type == "Gradient_Sigmoid":
        # Sharp sigmoid transition
        val = 1 / (1 + np.exp(-10 * (x_norm - 0.5)))
        
    elif pattern_type == "Bimodal_Distribution":
        # Two distinct regions with different base intensities
        region1 = (x < cx).astype(float) * 0.8
        region2 = (x >= cx).astype(float) * 0.3
        val = region1 + region2
        
    elif pattern_type == "Punctate_Dense":
        # Very dense small puncta - use pattern-specific seed
        rng = np.random.RandomState(pattern_seed)
        val = np.zeros(n)
        for _ in range(50):
            px = rng.uniform(x.min(), x.max())
            py = rng.uniform(y.min(), y.max())
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            val += np.exp(-dist**2 / (2 * 40**2))
        val = np.clip(val, 0, 1)
        
    elif pattern_type == "Periventricular":
        # Pattern around a central void (like around ventricles)
        void_mask = dist_norm < 0.15
        ring_val = np.exp(-((dist_norm - 0.25)**2) / (2 * 0.08**2))
        val = ring_val * ~void_mask
        
    elif pattern_type == "Asymmetric_Lobe":
        # Asymmetric lobular structure
        lobe1_x, lobe1_y = cx - width*0.2, cy
        lobe2_x, lobe2_y = cx + width*0.15, cy + height*0.1
        dist1 = np.sqrt((x - lobe1_x)**2 + (y - lobe1_y)**2)
        dist2 = np.sqrt((x - lobe2_x)**2 + (y - lobe2_y)**2)
        val = 0.7 * np.exp(-dist1**2 / (2 * (width*0.15)**2)) + \
              0.5 * np.exp(-dist2**2 / (2 * (width*0.1)**2))
        val = val / val.max()
        
    else:
        # Fallback: deterministic noise based on coordinates
        rng = np.random.RandomState(pattern_seed)
        val = rng.rand(n)
    
    # Optionally add small noise for realism (disabled by default for exact matching)
    if add_noise:
        if noise_seed is not None:
            rng = np.random.RandomState(noise_seed)
        else:
            rng = np.random.RandomState(pattern_seed + 999)
        val = val + rng.normal(0, 0.03, size=n)
    
    return np.clip(val, 0, 1)


def generate_realistic_noise(coords, noise_idx, gt_patterns_list, width=3000, height=3000):
    """
    Generate noise that is more similar to real patterns - 
    combinations, distortions, and partial matches of GT patterns.
    
    Uses deterministic seeding for reproducibility.
    """
    n = len(coords)
    x, y = coords[:, 0], coords[:, 1]
    
    # Use a deterministic seed based on noise_idx
    rng = np.random.RandomState(noise_idx * 7 + 42)
    
    noise_type = noise_idx % 10
    
    if noise_type == 0:
        # Mixed gradient with random direction
        theta = rng.uniform(0, 2*np.pi)
        x_rot = x * np.cos(theta) + y * np.sin(theta)
        val = (x_rot - x_rot.min()) / (x_rot.max() - x_rot.min() + 1e-8)
        
    elif noise_type == 1:
        # Partial blob (cropped/shifted version of GT blob)
        bx = rng.uniform(x.min() - 500, x.max() + 500)
        by = rng.uniform(y.min() - 500, y.max() + 500)
        sigma = rng.uniform(200, 600)
        dist = np.sqrt((x - bx)**2 + (y - by)**2)
        val = np.exp(-dist**2 / (2 * sigma**2))
        
    elif noise_type == 2:
        # Distorted stripes (different frequency than GT)
        freq = rng.uniform(50, 200)
        angle = rng.uniform(0, np.pi)
        proj = x * np.cos(angle) + y * np.sin(angle)
        val = (np.sin(proj / freq) + 1) / 2
        
    elif noise_type == 3:
        # Ring at wrong radius
        cx, cy = x.mean(), y.mean()
        dist = np.sqrt((x - cx)**2 + (y - cy)**2)
        target_r = rng.uniform(100, 800)
        sigma = rng.uniform(50, 150)
        val = np.exp(-((dist - target_r)**2) / (2 * sigma**2))
        
    elif noise_type == 4:
        # Combination of two random GT-like patterns
        p1_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p2_idx = rng.randint(0, min(20, len(gt_patterns_list)))
        p1 = generate_pattern_values(coords, gt_patterns_list[p1_idx], width, height)
        p2 = generate_pattern_values(coords, gt_patterns_list[p2_idx], width, height)
        w = rng.uniform(0.3, 0.7)
        val = w * p1 + (1 - w) * p2
        
    elif noise_type == 5:
        # Inverted version of a GT-like pattern
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height)
        val = 1 - base_pattern
        
    elif noise_type == 6:
        # Thresholded/binarized pattern
        p_idx = rng.randint(0, min(15, len(gt_patterns_list)))
        base_pattern = generate_pattern_values(coords, gt_patterns_list[p_idx], width, height)
        threshold = rng.uniform(0.3, 0.7)
        val = (base_pattern > threshold).astype(float)
        
    elif noise_type == 7:
        # Spatially varying noise (smooth random field)
        # Create low-frequency spatial noise
        freq_x = rng.uniform(200, 600)
        freq_y = rng.uniform(200, 600)
        phase_x = rng.uniform(0, 2*np.pi)
        phase_y = rng.uniform(0, 2*np.pi)
        val = (np.sin(x / freq_x + phase_x) * np.sin(y / freq_y + phase_y) + 1) / 2
        
    elif noise_type == 8:
        # Scattered spots with different spacing
        val = np.zeros(n)
        n_spots = rng.randint(10, 30)
        for _ in range(n_spots):
            sx = rng.uniform(x.min(), x.max())
            sy = rng.uniform(y.min(), y.max())
            sigma = rng.uniform(30, 80)
            dist = np.sqrt((x - sx)**2 + (y - sy)**2)
            val += np.exp(-dist**2 / (2 * sigma**2))
        val = np.clip(val, 0, 1)
        
    else:
        # Pure random but smoothed (correlated noise)
        val = rng.rand(n)
        # Add some spatial structure by mixing with coordinates
        val = 0.5 * val + 0.25 * (x - x.min()) / (x.max() - x.min() + 1e-8) + \
              0.25 * rng.rand(n)
    
    # Add extra random noise using the same rng
    val = val + rng.normal(0, 0.08, size=n)
    return np.clip(val, 0, 1)


def get_visium_hex_grid(width, height, spacing=100):
    """
    Generates coordinates for a Visium-like hexagonal grid.
    
    Visium specifications:
    - Spot diameter: 55 µm
    - Center-to-center spacing: 100 µm
    - Hexagonal arrangement
    
    Parameters:
    -----------
    width, height : float
        Dimensions of the area in micrometers
    spacing : float
        Center-to-center distance between spots (default 100 µm for Visium)
    
    Returns:
    --------
    numpy array of (x, y) coordinates
    """
    # For hexagonal grid:
    # - Horizontal spacing between spots in the same row = spacing
    # - Vertical spacing between rows = spacing * sqrt(3)/2
    # - Alternate rows are offset by spacing/2
    
    dy = spacing * np.sqrt(3) / 2  # Vertical distance between rows
    
    coords = []
    row = 0
    y = 0
    
    while y < height:
        # Offset for alternating rows
        x_offset = (spacing / 2) if row % 2 == 1 else 0
        x = x_offset
        
        while x < width:
            coords.append([x, y])
            x += spacing
        
        y += dy
        row += 1
    
    return np.array(coords)


def get_cartesian_grid(width, height, spacing):
    """Generates coordinates for a Cartesian grid."""
    x = np.arange(0, width, spacing)
    y = np.arange(0, height, spacing)
    xx, yy = np.meshgrid(x, y)
    return np.column_stack((xx.ravel(), yy.ravel()))


# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

# Field size - needs to accommodate brain dimensions (up to 5.4mm height, 5mm width)
# Using 6000x6000 µm field to give some margin
FIELD_WIDTH = 6000
FIELD_HEIGHT = 6000

print("=" * 60)
print("Generating Ground Truth Dummy Data with 50 PATTERNS")
print("Half-brain shapes, Visium-like RNA grid, no rotation")
print("=" * 60)
print(f"Groups: {GROUPS}")
print(f"GT Patterns: {len(GT_PATTERNS)}")
print(f"Noise Features: {NOISE_FEATURES}")
print(f"Field size: {FIELD_WIDTH}x{FIELD_HEIGHT} µm")
print(f"Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2")
print(f"RNA grid: Visium-like hexagonal, {RNA_SPOT_SPACING}µm spacing")
print(f"MSI grid: Cartesian, {MSI_PIXEL_SIZE}µm spacing")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

ground_truth_pairs = []

for group_idx, group in enumerate(GROUPS):
    for sample_idx in range(1, SAMPLES_PER_GROUP + 1):
        sample_id = f"{group}_{sample_idx}"
        seed = hash(sample_id) % 2**32
        print(f"  Processing {sample_id}...")
        
        # 1. Define Half-Brain Tissue Shape
        tissue_fn = generate_halfbrain_shape(seed, width=FIELD_WIDTH, height=FIELD_HEIGHT)
        
        # 2. Generate MSI Data (Cartesian grid)
        raw_msi_coords = get_cartesian_grid(FIELD_WIDTH, FIELD_HEIGHT, MSI_PIXEL_SIZE)
        mask_msi = [tissue_fn(x, y) for x, y in raw_msi_coords]
        msi_coords = raw_msi_coords[mask_msi]
        
        n_total_features = len(GT_PATTERNS) + NOISE_FEATURES
        msi_data = np.zeros((len(msi_coords), n_total_features))
        var_names_msi = []
        
        # Generate GT pattern features
        for i, pat in enumerate(GT_PATTERNS):
            msi_data[:, i] = generate_pattern_values(msi_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT)
            var_names_msi.append(f"MZ_{pat}")
        
        # Generate realistic noise features
        for i in range(NOISE_FEATURES):
            msi_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                msi_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT
            )
            var_names_msi.append(f"MZ_Noise_{i}")
            
        obs_msi = pd.DataFrame(index=[f"spot_{i}" for i in range(len(msi_coords))])
        obs_msi['x_um'] = msi_coords[:, 0]
        obs_msi['y_um'] = msi_coords[:, 1]
        
        adata_msi = ad.AnnData(X=msi_data, obs=obs_msi)
        adata_msi.var_names = var_names_msi
        adata_msi.write(os.path.join(MSI_DIR, f"halfbrain_{group.lower()}_{sample_idx}_filtered_common.h5ad"))
        
        
        # 3. Generate RNA Data (Visium-like hexagonal grid)
        raw_rna_coords = get_visium_hex_grid(FIELD_WIDTH, FIELD_HEIGHT, spacing=RNA_SPOT_SPACING)
        mask_rna = [tissue_fn(x, y) for x, y in raw_rna_coords]
        rna_coords = raw_rna_coords[mask_rna]
        
        rna_data = np.zeros((len(rna_coords), n_total_features))
        var_names_rna = []
        
        # Generate GT pattern features (NO ROTATION - same coordinates as mask)
        for i, pat in enumerate(GT_PATTERNS):
            rna_data[:, i] = generate_pattern_values(rna_coords, pat, width=FIELD_WIDTH, height=FIELD_HEIGHT)
            var_names_rna.append(f"Gene_{pat}")
            
            if group_idx == 0 and sample_idx == 1:
                ground_truth_pairs.append((f"Gene_{pat}", f"MZ_{pat}"))
        
        # Generate realistic noise features
        for i in range(NOISE_FEATURES):
            rna_data[:, len(GT_PATTERNS) + i] = generate_realistic_noise(
                rna_coords, i, GT_PATTERNS, width=FIELD_WIDTH, height=FIELD_HEIGHT
            )
            var_names_rna.append(f"Gene_Noise_{i}")
        
        # Store coordinates directly (NO rotation/transformation)
        obs_rna = pd.DataFrame(index=[f"spot_{i}" for i in range(len(rna_coords))])
        obs_rna['x_um'] = rna_coords[:, 0]
        obs_rna['y_um'] = rna_coords[:, 1]
        
        adata_rna = ad.AnnData(X=rna_data, obs=obs_rna)
        adata_rna.var_names = var_names_rna
        adata_rna.write(os.path.join(RNA_DIR, f"{group}_{sample_idx}.h5ad"))

print("\n" + "=" * 60)
print("DONE! Data generated.")
print("=" * 60)
print("\n50 GROUND TRUTH MATCHES:")
print("-" * 60)
for i, (gene, mz) in enumerate(ground_truth_pairs, 1):
    print(f"  {i:2d}. {gene:<30}  <==>  {mz}")
print("-" * 60)
print(f"\nTotal features per sample: {len(GT_PATTERNS)} GT + {NOISE_FEATURES} Noise = {len(GT_PATTERNS) + NOISE_FEATURES}")

Generating Ground Truth Dummy Data with 50 PATTERNS
Half-brain shapes, Visium-like RNA grid, no rotation
Groups: ['YC', 'YAD', 'AC', 'AAD']
GT Patterns: 50
Noise Features: 500
Field size: 6000x6000 µm
Brain dimensions: H=4.9-5.4mm, W=4.4-5.0mm, H/W=1.0-1.2
RNA grid: Visium-like hexagonal, 100µm spacing
MSI grid: Cartesian, 60µm spacing
Output: dummy_data_50
  Processing YC_1...
  Processing YC_2...
  Processing YC_3...
  Processing YC_4...
  Processing YAD_1...
  Processing YAD_2...
  Processing YAD_3...
  Processing YAD_4...
  Processing AC_1...
  Processing AC_2...
  Processing AC_3...
  Processing AC_4...
  Processing AAD_1...
  Processing AAD_2...
  Processing AAD_3...
  Processing AAD_4...

DONE! Data generated.

50 GROUND TRUTH MATCHES:
------------------------------------------------------------
   1. Gene_Gradient_X                 <==>  MZ_Gradient_X
   2. Gene_Gradient_Y                 <==>  MZ_Gradient_Y
   3. Gene_Gradient_Diagonal_NE       <==>  MZ_Gradient_Diagonal_NE
  